# Model Optimization Pipeline (Single Model Mode)

This notebook orchestrates the complete model optimization workflow:
1. **Load Base Model** - Load PyTorch model from HuggingFace (FP32)
2. **Wanda Pruning** - 30% sparsity with WikiText-2 calibration, then export to ONNX
3. **Quantization** - INT8 (applied to pruned ONNX model ONLY)
4. **Evaluation** - Comprehensive quality and performance metrics
5. **Comparison Report** - Summary and visualization of results

## 🎯 Evaluation Methodology

### Step A: Static Quality Metrics
**LLM (Language Models):**
- WikiText-2 Perplexity (PPL)
- Optional: lm-evaluation-harness for MMLU, GSM8K accuracy

**VLM (Vision-Language Models):**
- COCO BLEU-4 & CIDEr scores
- Optional: VLMEvalKit for MM-Bench, MME benchmarks

### Step B: Distribution Alignment
**KL Divergence Testing:**
- Compares token probability distributions between quantized and FP16 reference model
- Lower KL divergence = safer quantization
- Measured on WikiText-2 prompts

## 🚀 Checkpoint System
**The pipeline now includes automatic checkpointing!**
- If pruning has completed before, it will skip pruning and use the saved model
- If ONNX export has completed before, it will skip export and use the saved model
- If quantization has completed before, it will skip quantization and use the saved model
- Set `skip_existing=False` in the code to force rerun any step

This means if ONNX export fails, you can fix it and rerun without redoing the expensive pruning step!

## Available Models
- `google/gemma-2-2b` (2B params, ~4GB)
- `meta-llama/Llama-3.2-3B-Instruct` (3B params, ~6GB)
- `deepseek-ai/DeepSeek-R1-Distill-Qwen-7B` (7B params, ~14GB) ⚠️ Large!
- `lmms-lab/LLaVA-OneVision-1.5-4B-Instruct` (4B params, multimodal) ⚠️ Special handling

**💡 This notebook processes ONE model at a time to avoid OOM issues.**

## What is Wanda Pruning?
Wanda (Pruning by Weights And activations) is an advanced pruning method that:
- Uses WikiText-2 calibration data to measure activation importance
- Computes importance scores as: **importance = |weight| × |activation|**
- Prunes weights with lowest importance (not just smallest magnitude)
- Better preserves model quality compared to magnitude-only pruning

## Complete Workflow
```
Step 1: Load PyTorch Model (FP32) - Base model loaded for reference
Step 2: Prune with Wanda (WikiText-2 calibration) → Export pruned model to ONNX
        ⚡ Checkpointed: Skips if already done
Step 3: Quantize pruned ONNX model ONLY (NOT base model)
        ⚡ Checkpointed: Skips if already done
  - Pruned ONNX (base)
  - Pruned ONNX → INT8
Step 4: Evaluate all variants
  - Step A: Static quality metrics (Perplexity, BLEU-4, CIDEr, etc.)
  - Step B: KL divergence vs FP16 reference model
Step 5: Generate comparison report
  
Result: 3 model variants for comparison (INT4 disabled for now)
```

## Setup and Configuration

In [ ]:
import os
import sys
import json
from pathlib import Path
from dataclasses import asdict
import warnings
warnings.filterwarnings('ignore')

# Import our custom modules
from quantization import ModelQuantizer
from pruning import ModelPruner
from evaluation import ModelEvaluator, EvaluationMetrics

print("✓ Modules imported successfully")

In [ ]:
# ========================================
# 🎯 SELECT MODEL TO PROCESS
# ========================================

# Choose ONE model to process (uncomment the one you want):
# SELECTED_MODEL = 'google/gemma-2-2b'  # Start with smallest model
SELECTED_MODEL = 'meta-llama/Llama-3.2-3B-Instruct'
# SELECTED_MODEL = 'deepseek-ai/DeepSeek-R1-Distill-Qwen-7B'  # ⚠️ Requires ~14GB RAM
# SELECTED_MODEL = 'lmms-lab/LLaVA-OneVision-1.5-4B-Instruct'  # ⚠️ Multimodal

# Configuration - NO SPACES IN PATH!
OUTPUT_DIR = "/Users/sashalai/Documents/UW/26sp/eep564/FinalProject/model_quantization/model_quantization_outputs"

# Pruning configuration - Using 30% sparsity for better quality preservation
SPARSITY_LEVEL = 0.3  # 30% pruning

# Number of evaluation runs for timing
NUM_EVAL_RUNS = 10

# Display configuration
print("="*80)
print("WANDA PRUNING OPTIMIZATION WORKFLOW")
print("="*80)
print(f"Selected model: {SELECTED_MODEL}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Pruning sparsity: {SPARSITY_LEVEL * 100:.0f}%")
print(f"Evaluation runs: {NUM_EVAL_RUNS}")
print("\nWorkflow:")
print("  1. Load base PyTorch model (FP32)")
print("  2. Wanda Prune (WikiText-2) → Export pruned model to ONNX")
print("  3. Quantize pruned ONNX (INT8 only for now)")
print("  4. Evaluate variants (base PyTorch + pruned ONNX + pruned INT8)")
print("  5. Generate comparison report")
print("\nModel Variants:")
print("  - Base PyTorch (FP32)")
print("  - Pruned ONNX (FP32)")
print("  - Pruned ONNX + INT8")
print("\nPruning Method: Wanda (Weights AND Activations)")
print("  - Importance = |weight| × |activation|")
print("  - Calibration: WikiText-2 (64 samples for faster processing)")
print("\nExpected time: ~30-40 minutes for gemma-2-2b, ~40-60 minutes for Llama-3.2-3B")
print("="*80)

## Step 1: Load Base Model

Load the original PyTorch model from HuggingFace in FP32 precision.

This step:
1. Loads the PyTorch model and tokenizer
2. Stores them for later evaluation and comparison
3. Does NOT export to ONNX (we only export after pruning)

**Note:** The base PyTorch model will be used as a baseline for comparison.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

print(f"\n{'='*80}")
print(f"STEP 1: LOAD BASE MODEL")
print(f"Processing: {SELECTED_MODEL}")
print(f"{'='*80}\n")

# Define model short name
model_short_name = SELECTED_MODEL.split('/')[-1]

try:
    print("Loading PyTorch model from HuggingFace...")
    
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(
        SELECTED_MODEL,
        trust_remote_code=True
    )
    
    # Load model in FP32
    base_model = AutoModelForCausalLM.from_pretrained(
        SELECTED_MODEL,
        torch_dtype=torch.float32,
        trust_remote_code=True,
        device_map="cpu"
    )
    
    print(f"✓ Model loaded successfully!")
    print(f"✓ Model parameters: {sum(p.numel() for p in base_model.parameters()):,}")
    print(f"✓ Model dtype: {base_model.dtype}")
    
    base_model_loaded = True
    
except Exception as e:
    print(f"✗ Failed to load model: {str(e)}")
    base_model_loaded = False
    base_model = None
    tokenizer = None

print(f"\n{'='*80}")
print("BASE MODEL LOADING RESULTS")
print(f"{'='*80}")
print(f"Model loaded: {'✓' if base_model_loaded else '✗'}")
print(f"Ready for pruning: {'✓' if base_model_loaded else '✗'}")
print(f"{'='*80}")

print("\nNote: Base model is loaded in memory but NOT exported to ONNX.")
print("      We will only export to ONNX after pruning (Step 2).")

## Step 2: Wanda Pruning (Correct Workflow)

**CORRECT PRUNING WORKFLOW:**
1. Load PyTorch model in FP32 (high precision for calculations)
2. Load WikiText-2 calibration data from HuggingFace datasets
3. Apply **Wanda pruning** - pruning by weights and activations (30% sparsity)
   - Collects activation statistics using calibration data
   - Computes importance scores: importance = |weight| × |activation|
   - Prunes weights with lowest importance scores
4. Export pruned PyTorch model to ONNX

**Why Wanda pruning?**
- More accurate than magnitude-only pruning
- Uses actual activation patterns from real data (WikiText-2)
- Better preserves model quality at moderate sparsity levels
- Prunes unimportant weights based on both weights AND activations

**Why 30% sparsity?**
- Better quality preservation than 50%
- Still provides significant compression
- Good balance between model size and accuracy

**Next step:** Quantize the pruned ONNX model (next cell)

In [ ]:
# Initialize pruner
pruner = ModelPruner(OUTPUT_DIR)

print(f"\n{'='*80}")
print(f"STEP 2: PRUNING")
print(f"Processing: {SELECTED_MODEL} with {SPARSITY_LEVEL*100:.0f}% sparsity")
print(f"{'='*80}\n")

# Run pruning pipeline with ONNX export enabled (correct workflow)
# Note: skip_existing=True means it will use checkpoints if available
pruning_result = pruner.process_model(
    SELECTED_MODEL,
    sparsity=SPARSITY_LEVEL,
    pruning_method="wanda",  # Wanda pruning with WikiText-2 calibration
    export_onnx=True,  # REQUIRED for correct workflow: prune → export → quantize
    num_calibration_samples=64,  # Number of WikiText-2 samples (reduced from 128 for speed)
    device="cpu",  # Use "cuda" if GPU available
    skip_existing=True  # Use checkpoints if available (set False to force rerun)
)

# Print summary
print(f"\n{'='*80}")
print("PRUNING RESULTS")
print(f"{'='*80}")
print(f"Success: {'✓' if pruning_result['success'] else '✗'}")
if pruning_result.get('checkpoints_used'):
    print(f"Checkpoints used: {', '.join(pruning_result['checkpoints_used'])}")
if pruning_result['sparsity_stats']:
    stats = pruning_result['sparsity_stats']
    print(f"Achieved Sparsity: {stats['overall_sparsity']*100:.2f}%")
    print(f"Zero Parameters: {stats['zero_params']:,} / {stats['total_params']:,}")
if pruning_result['pytorch_model_path']:
    print(f"PyTorch Model: {pruning_result['pytorch_model_path']}")
if pruning_result['onnx_model_path']:
    print(f"ONNX Model: {pruning_result['onnx_model_path']}")
if pruning_result['errors']:
    print(f"Errors: {', '.join(pruning_result['errors'])}")
print(f"{'='*80}")

In [ ]:
# Save pruning results
model_short_name = SELECTED_MODEL.split('/')[-1]
prune_result_path = Path(OUTPUT_DIR) / f"pruning_result_{model_short_name}.json"

with open(prune_result_path, 'w') as f:
    json.dump(pruning_result, f, indent=2)

print(f"✓ Pruning results saved to: {prune_result_path}")

## Step 3: Quantization (Pruned Model Only)

Quantize the pruned ONNX model to INT8.

This creates 2 pruned model variants:
1. **pruned_30pct** - Pruned ONNX model (FP32)
2. **pruned_30pct_int8** - Pruned + INT8 quantization

**Note:** 
- The base PyTorch model (FP32) will also be evaluated for comparison.
- INT4 quantization is currently disabled for testing INT8 first.

In [ ]:
# Initialize quantizer
quantizer = ModelQuantizer(OUTPUT_DIR)

print(f"\n{'='*80}")
print(f"STEP 3: QUANTIZATION (PRUNED MODEL)")
print(f"{'='*80}\n")

# Dictionary to store all quantized model paths
all_model_paths = {}

# Store base PyTorch model for evaluation
if base_model_loaded:
    all_model_paths['base_pytorch'] = 'base_model'  # Reference to in-memory model
    print("✓ Base PyTorch model available for evaluation")
else:
    print("⚠️ Base PyTorch model not available")

# ===== QUANTIZE PRUNED MODEL =====
print("\n" + "="*80)
print("QUANTIZING PRUNED ONNX MODEL (INT8 ONLY)")
print("="*80)

if pruning_result['success'] and pruning_result.get('onnx_model_path'):
    pruned_onnx_path = Path(pruning_result['onnx_model_path'])
    
    print(f"\nPruned ONNX source: {pruned_onnx_path}")
    
    # Store the original pruned ONNX path
    all_model_paths['pruned_onnx'] = str(pruned_onnx_path)
    
    # Quantize to INT8 (with checkpoint detection)
    print("\n[1/1] Quantizing pruned model to INT8...")
    print("Note: INT4 is disabled for now - focusing on INT8 testing first")
    pruned_int8_path = quantizer.quantize_onnx_from_path(
        pruned_onnx_path,
        "int8",
        skip_existing=True  # Use checkpoint if available
    )
    if pruned_int8_path:
        all_model_paths['pruned_int8'] = str(pruned_int8_path)
        print(f"✓ Pruned INT8 complete: {pruned_int8_path}")
    else:
        print("✗ Pruned INT8 quantization failed")
    
    # INT4 is currently disabled
    all_model_paths['pruned_int4'] = None
else:
    print("⚠️ Pruning did not complete successfully or ONNX export failed.")
    print("   Cannot quantize pruned model.")
    all_model_paths['pruned_onnx'] = None
    all_model_paths['pruned_int8'] = None
    all_model_paths['pruned_int4'] = None

# ===== SUMMARY =====
print(f"\n{'='*80}")
print("QUANTIZATION SUMMARY")w
print(f"{'='*80}")
print("Base Model:")
print(f"  Base PyTorch (FP32): {'✓' if all_model_paths.get('base_pytorch') else '✗'}")
print("\nPruned Model Variants:")
print(f"  Pruned ONNX (FP32): {'✓' if all_model_paths.get('pruned_onnx') else '✗'}")
print(f"  Pruned INT8:         {'✓' if all_model_paths.get('pruned_int8') else '✗'}")
print(f"  Pruned INT4:         Disabled (testing INT8 first)")
print(f"\nTotal variants: {sum(1 for v in all_model_paths.values() if v)}/3")
print(f"{'='*80}")

# Store for evaluation step
pruned_model_paths = {
    'pruned_onnx': all_model_paths.get('pruned_onnx'),
    'pruned_int8': all_model_paths.get('pruned_int8'),
    'pruned_int4': None  # Disabled
}

## Step 4: Evaluation

Evaluate all 4 model variants:

**Base model:**
1. Base PyTorch (FP32) - serves as reference for KL divergence

**Pruned variants:**
2. Pruned ONNX (FP32)
3. Pruned ONNX + INT8
4. Pruned ONNX + INT4

**Evaluation Methodology:**

### Step A: Static Quality Metrics

**For LLMs (Language Models):**
- **Perplexity on WikiText-2**: Measures how well the model predicts text (lower is better)
- **Optional: lm-evaluation-harness**: MMLU and GSM8K accuracy scores

**For VLMs (Vision-Language Models):**
- **COCO BLEU-4 & CIDEr**: Measures caption quality on COCO dataset
- **Optional: VLMEvalKit**: MM-Bench and MME benchmark scores

### Step B: Distribution Alignment (KL Divergence)

Compares token probability distributions between:
- **Reference model**: Original FP16/FP32 model
- **Quantized models**: Pruned, INT8, and INT4 variants

Lower KL divergence = safer quantization (less distribution shift)

**Other metrics measured:**
- Inference speed (mean and std)
- Throughput (tokens/second)
- Model size (MB)

In [ ]:
# Initialize evaluator
evaluator = ModelEvaluator(OUTPUT_DIR)

model_metrics = []

print(f"\n{'='*80}")
print(f"STEP 4: EVALUATION")
print(f"Evaluating: {SELECTED_MODEL}")
print(f"{'='*80}\n")

# Determine if this is a VLM (Vision-Language Model)
is_vlm = 'llava' in SELECTED_MODEL.lower() or 'vision' in SELECTED_MODEL.lower()

# Evaluation configuration
EVALUATE_LM_HARNESS = False  # Set to True to run MMLU/GSM8K (takes longer)
EVALUATE_COCO = False  # Set to True for VLM COCO evaluation
EVALUATE_VLMKIT = False  # Set to True for VLM VLMEvalKit evaluation
NUM_PERPLEXITY_SAMPLES = 50
NUM_KLD_PROMPTS = 20

print(f"Model type: {'VLM (Vision-Language Model)' if is_vlm else 'LLM (Language Model)'}")
print(f"\nEvaluation plan:")
print(f"  - Step A (Static Metrics):")
if is_vlm:
    print(f"    - COCO BLEU-4 & CIDEr: {'Enabled' if EVALUATE_COCO else 'Disabled'}")
    print(f"    - VLMEvalKit (MM-Bench, MME): {'Enabled' if EVALUATE_VLMKIT else 'Disabled'}")
else:
    print(f"    - WikiText-2 Perplexity: Enabled")
    print(f"    - lm-evaluation-harness (MMLU, GSM8K): {'Enabled' if EVALUATE_LM_HARNESS else 'Disabled'}")
print(f"  - Step B (Distribution Alignment):")
print(f"    - KL Divergence vs FP16 model: Enabled")
print()

# ===== EVALUATE BASE PYTORCH MODEL =====
print("BASE MODEL (PyTorch FP32)")
print("-"*80)

base_model_metrics = None
if base_model_loaded and base_model is not None:
    print("[1/4] Evaluating base PyTorch model...")
    print("  Note: This serves as the reference (FP16) model for KL divergence comparison")
    
    # Create temporary directory for base model evaluation
    import tempfile
    temp_dir = Path(tempfile.mkdtemp())
    temp_model_path = temp_dir / "base_model"
    
    # Save base model temporarily
    base_model.save_pretrained(temp_model_path)
    tokenizer.save_pretrained(temp_model_path)
    
    base_model_metrics = evaluator.evaluate_model(
        model_path=temp_model_path,
        model_name=SELECTED_MODEL,
        model_type="base",
        num_runs=NUM_EVAL_RUNS,
        reference_model=None,  # Base model has no reference
        reference_tokenizer=None,
        evaluate_quality=True,
        evaluate_kld=False,  # Don't calculate KLD for base model
        evaluate_lm_harness=EVALUATE_LM_HARNESS,
        is_vlm=is_vlm,
        evaluate_coco=EVALUATE_COCO,
        evaluate_vlmkit=EVALUATE_VLMKIT,
        num_perplexity_samples=NUM_PERPLEXITY_SAMPLES
    )
    
    if base_model_metrics:
        model_metrics.append(base_model_metrics)
        print(f"✓ Base model evaluation complete")
    
    # Clean up temp directory
    import shutil
    shutil.rmtree(temp_dir)
else:
    print("[1/4] ⚠️ Base model not loaded")

# ===== PRUNED VARIANTS =====
print("\n\nPRUNED VARIANTS (Pruned ONNX + Quantization)")
print("-"*80)

sparsity_str = f"{int(SPARSITY_LEVEL * 100)}pct"

# 2. Evaluate pruned ONNX
if pruned_model_paths.get('pruned_onnx'):
    pruned_onnx = Path(pruned_model_paths['pruned_onnx'])
    if pruned_onnx.exists():
        print(f"[2/4] Evaluating pruned {SPARSITY_LEVEL*100:.0f}% ONNX model...")
        metrics = evaluator.evaluate_model(
            model_path=pruned_onnx,
            model_name=SELECTED_MODEL,
            model_type=f"pruned_{sparsity_str}",
            num_runs=NUM_EVAL_RUNS,
            reference_model=base_model if base_model_loaded else None,
            reference_tokenizer=tokenizer if base_model_loaded else None,
            evaluate_quality=True,
            evaluate_kld=True if base_model_loaded else False,
            evaluate_lm_harness=EVALUATE_LM_HARNESS,
            is_vlm=is_vlm,
            evaluate_coco=EVALUATE_COCO,
            evaluate_vlmkit=EVALUATE_VLMKIT,
            num_perplexity_samples=NUM_PERPLEXITY_SAMPLES,
            num_kld_prompts=NUM_KLD_PROMPTS
        )
        if metrics:
            model_metrics.append(metrics)
            print(f"✓ Pruned ONNX evaluation complete")
    else:
        print(f"[2/4] ⚠️ Pruned ONNX not found at {pruned_onnx}")
else:
    print(f"[2/4] ⚠️ Pruned ONNX not available")

# 3. Evaluate pruned INT8
if pruned_model_paths.get('pruned_int8'):
    pruned_int8 = Path(pruned_model_paths['pruned_int8'])
    if pruned_int8.exists():
        print(f"\n[3/4] Evaluating pruned {SPARSITY_LEVEL*100:.0f}% + INT8...")
        metrics = evaluator.evaluate_model(
            model_path=pruned_int8,
            model_name=SELECTED_MODEL,
            model_type=f"pruned_{sparsity_str}_int8",
            num_runs=NUM_EVAL_RUNS,
            reference_model=base_model if base_model_loaded else None,
            reference_tokenizer=tokenizer if base_model_loaded else None,
            evaluate_quality=True,
            evaluate_kld=True if base_model_loaded else False,
            evaluate_lm_harness=EVALUATE_LM_HARNESS,
            is_vlm=is_vlm,
            evaluate_coco=EVALUATE_COCO,
            evaluate_vlmkit=EVALUATE_VLMKIT,
            num_perplexity_samples=NUM_PERPLEXITY_SAMPLES,
            num_kld_prompts=NUM_KLD_PROMPTS
        )
        if metrics:
            model_metrics.append(metrics)
            print(f"✓ Pruned INT8 evaluation complete")
    else:
        print(f"\n[3/4] ⚠️ Pruned INT8 not found at {pruned_int8}")
else:
    print(f"\n[3/4] ⚠️ Pruned INT8 not available")

# 4. Evaluate pruned INT4
if pruned_model_paths.get('pruned_int4'):
    pruned_int4 = Path(pruned_model_paths['pruned_int4'])
    if pruned_int4.exists():
        print(f"\n[4/4] Evaluating pruned {SPARSITY_LEVEL*100:.0f}% + INT4...")
        metrics = evaluator.evaluate_model(
            model_path=pruned_int4,
            model_name=SELECTED_MODEL,
            model_type=f"pruned_{sparsity_str}_int4",
            num_runs=NUM_EVAL_RUNS,
            reference_model=base_model if base_model_loaded else None,
            reference_tokenizer=tokenizer if base_model_loaded else None,
            evaluate_quality=True,
            evaluate_kld=True if base_model_loaded else False,
            evaluate_lm_harness=EVALUATE_LM_HARNESS,
            is_vlm=is_vlm,
            evaluate_coco=EVALUATE_COCO,
            evaluate_vlmkit=EVALUATE_VLMKIT,
            num_perplexity_samples=NUM_PERPLEXITY_SAMPLES,
            num_kld_prompts=NUM_KLD_PROMPTS
        )
        if metrics:
            model_metrics.append(metrics)
            print(f"✓ Pruned INT4 evaluation complete")
    else:
        print(f"\n[4/4] ⚠️ Pruned INT4 not found at {pruned_int4}")
else:
    print(f"\n[4/4] ⚠️ Pruned INT4 not available")

print(f"\n{'='*80}")
print(f"EVALUATION COMPLETED: {len(model_metrics)} variants evaluated")
print(f"{'='*80}")

# Print evaluation summary
if model_metrics:
    print("\nEVALUATION SUMMARY")
    print("-"*80)
    print(f"{'Model Type':<25} {'Perplexity':<12} {'KL Div':<12} {'Size (MB)':<12}")
    print("-"*80)
    for m in model_metrics:
        ppl_str = f"{m.perplexity:.2f}" if m.perplexity else "N/A"
        kld_str = f"{m.kl_divergence:.6f}" if m.kl_divergence else "N/A"
        size_str = f"{m.model_size_mb:.2f}" if m.model_size_mb else "N/A"
        print(f"{m.model_type:<25} {ppl_str:<12} {kld_str:<12} {size_str:<12}")
    print("-"*80)


In [ ]:
# Save evaluation metrics for this model
eval_result_path = Path(OUTPUT_DIR) / "evaluation" / f"metrics_{model_short_name}.json"
eval_result_path.parent.mkdir(parents=True, exist_ok=True)

with open(eval_result_path, 'w') as f:
    json.dump([asdict(m) for m in model_metrics], f, indent=2)

print(f"✓ Evaluation metrics saved to: {eval_result_path}")

## Step 5: Comparison Report

Generate a comparison report for this model showing:
- Size reductions
- Speedup factors
- Throughput comparisons

In [ ]:
# Generate comparison report for this model
report_path = Path(OUTPUT_DIR) / "evaluation" / f"comparison_{model_short_name}.json"

report = evaluator.generate_comparison_report(
    all_metrics=model_metrics,
    output_path=report_path
)

print(f"✓ Comparison report saved to: {report_path}")

In [ ]:
# Display formatted comparison table
if report and 'details' in report and report['details']:
    evaluator.print_comparison_table(report)
else:
    print("⚠️ No comparison data available. Make sure models were evaluated successfully.")

## Visualization (Optional)

Create visualizations to compare variants of this model

In [ ]:
# Visualize results for this model
try:
    import matplotlib.pyplot as plt
    import numpy as np
    
    if not report or 'details' not in report or not report['details']:
        print("⚠️ No data to visualize")
    else:
        # Get data for the selected model
        model_data = report['details'].get(SELECTED_MODEL)
        
        if not model_data:
            print(f"⚠️ No data found for {SELECTED_MODEL}")
        else:
            variants = ['Base'] + [v['type'] for v in model_data['variants']]
            sizes = [model_data['base_size_mb']] + [v['size_mb'] for v in model_data['variants']]
            times = [model_data['base_inference_time']] + [v['inference_time'] for v in model_data['variants']]
            throughputs = [model_data['base_size_mb'] / model_data['base_inference_time']] + [v['throughput'] for v in model_data['variants']]
            
            fig, axes = plt.subplots(1, 3, figsize=(18, 5))
            
            # Plot 1: Model size comparison
            axes[0].bar(variants, sizes, color='steelblue', alpha=0.7)
            axes[0].set_ylabel('Model Size (MB)', fontsize=12)
            axes[0].set_title(f'Model Size Comparison\n{model_short_name}', fontsize=14, fontweight='bold')
            axes[0].tick_params(axis='x', rotation=45)
            axes[0].grid(axis='y', alpha=0.3)
            
            # Plot 2: Inference time comparison
            axes[1].bar(variants, times, color='coral', alpha=0.7)
            axes[1].set_ylabel('Inference Time (seconds)', fontsize=12)
            axes[1].set_title(f'Inference Speed Comparison\n{model_short_name}', fontsize=14, fontweight='bold')
            axes[1].tick_params(axis='x', rotation=45)
            axes[1].grid(axis='y', alpha=0.3)
            
            # Plot 3: Throughput comparison
            axes[2].bar(variants, throughputs, color='seagreen', alpha=0.7)
            axes[2].set_ylabel('Throughput (tokens/sec)', fontsize=12)
            axes[2].set_title(f'Throughput Comparison\n{model_short_name}', fontsize=14, fontweight='bold')
            axes[2].tick_params(axis='x', rotation=45)
            axes[2].grid(axis='y', alpha=0.3)
            
            plt.tight_layout()
            
            # Save figure
            fig_path = Path(OUTPUT_DIR) / "evaluation" / f"comparison_chart_{model_short_name}.png"
            plt.savefig(fig_path, dpi=300, bbox_inches='tight')
            print(f"✓ Visualization saved to: {fig_path}")
            
            plt.show()
    
except ImportError:
    print("⚠️ matplotlib not installed. Skipping visualization.")
    print("   Install with: pip install matplotlib")

## Summary

Review the key findings for this model

In [ ]:
print("\n" + "="*80)
print("OPTIMIZATION PIPELINE SUMMARY")
print("="*80)

print(f"\nModel: {SELECTED_MODEL}")
print(f"Total variants evaluated: {len(model_metrics)}")

print("\nOutput files:")
print(f"  - Quantization results: {quant_result_path}")
print(f"  - Pruning results: {prune_result_path}")
print(f"  - Evaluation metrics: {eval_result_path}")
print(f"  - Comparison report: {report_path}")

if report and 'details' in report and SELECTED_MODEL in report['details']:
    model_data = report['details'][SELECTED_MODEL]
    
    print("\nKey Insights:")
    print(f"  Base model size: {model_data['base_size_mb']:.2f} MB")
    print(f"  Base inference time: {model_data['base_inference_time']:.4f}s")
    
    if model_data['variants']:
        print("\n  Optimization Results:")
        for variant in model_data['variants']:
            print(f"    {variant['type']:15s}: {variant['size_mb']:8.2f} MB ({variant['size_reduction_pct']:5.1f}% smaller) | "
                  f"{variant['speedup']:.2f}x faster | {variant['throughput']:.2f} tokens/sec")
        
        best_compression = max(model_data['variants'], key=lambda x: x['size_reduction_pct'])
        best_speedup = max(model_data['variants'], key=lambda x: x['speedup'])
        
        print(f"\n  🏆 Best compression: {best_compression['type']} ({best_compression['size_reduction_pct']:.1f}% reduction)")
        print(f"  🏆 Best speedup: {best_speedup['type']} ({best_speedup['speedup']:.2f}x faster)")
else:
    print("\n⚠️ No comparison data available")

print("\n" + "="*80)
print("✓ Pipeline completed successfully!")
print("="*80)
print("\n💡 To process another model:")
print("   1. Change SELECTED_MODEL in the configuration cell")
print("   2. Run all cells again (Kernel > Restart & Run All)")

---

## Optional: Aggregate Results Across All Models

If you've run this notebook for multiple models, you can aggregate all results here.

In [ ]:
# Aggregate all model results
import glob

eval_dir = Path(OUTPUT_DIR) / "evaluation"
all_metrics_files = glob.glob(str(eval_dir / "metrics_*.json"))

if all_metrics_files:
    print(f"Found {len(all_metrics_files)} evaluation result files:\n")
    
    all_aggregated_metrics = []
    
    for metrics_file in all_metrics_files:
        with open(metrics_file, 'r') as f:
            metrics = json.load(f)
            all_aggregated_metrics.extend(metrics)
        print(f"  ✓ {Path(metrics_file).name}")
    
    # Generate combined report
    combined_report_path = eval_dir / "comparison_all_models.json"
    
    # Convert to EvaluationMetrics objects
    metrics_objects = [EvaluationMetrics(**m) for m in all_aggregated_metrics]
    
    combined_report = evaluator.generate_comparison_report(
        all_metrics=metrics_objects,
        output_path=combined_report_path
    )
    
    print(f"\n✓ Combined report saved to: {combined_report_path}")
    
    # Display combined comparison
    print("\n" + "="*80)
    print("COMBINED COMPARISON ACROSS ALL MODELS")
    print("="*80)
    evaluator.print_comparison_table(combined_report)
else:
    print("No evaluation results found yet. Run the pipeline for at least one model first.")